In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import math
from nltk.translate.bleu_score import corpus_bleu
from tqdm import tqdm
import os
from torch.utils.data import DataLoader,Dataset
import time
import nltk
from collections import Counter
import torch.optim as optim


from PIL import Image
import torchvision.transforms as transforms

In [ ]:
class ResNetEncoder(nn.Module):
    def __init__(self, resnet_version='resnet50'):
        super().__init__()
        # Load a pre-trained ResNet model
        resnet = getattr(models, resnet_version)(pretrained=True)
        # Remove the fully connected layers to use as a feature extractor
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-2])
        # Ensure output is 7×7 spatially (after the last convolutional block)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((7,7))

    def forward(self, images):
        """
        images: (B, 3, 224, 224)
        returns: (B, 49, 2048)  — 49 spatial locations × 2048-dim features
        """
        feats = self.feature_extractor(images)  # (B, 2048, H, W)
        feats = self.adaptive_pool(feats)       # (B, 2048, 7, 7)
        B, C, H, W = feats.shape
        # Flatten spatial dims and move channels last
        feats = feats.view(B, C, H * W).permute(0, 2, 1)  # (B, 49, 2048)
        return feats


In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim, feature_dim, attn_dim):
        super().__init__()
        self.W_h = nn.Linear(hidden_dim, attn_dim)
        self.W_f = nn.Linear(feature_dim, attn_dim)
        self.v   = nn.Linear(attn_dim, 1)

    def forward(self, hidden, features):
        """
        hidden:   (B, hidden_dim)  — previous hidden state
        features: (B, 49, feature_dim)
        returns:
          context: (B, feature_dim)  — weighted sum of features
          alpha:   (B, 49)           — attention weights
        """
        # Project hidden and features
        h = self.W_h(hidden).unsqueeze(1)           # (B, 1, attn_dim)
        f = self.W_f(features)                      # (B, 49, attn_dim)
        # Score each spatial location
        scores = self.v(torch.tanh(h + f)).squeeze(2)  # (B, 49)
        alpha  = F.softmax(scores, dim=1)             # (B, 49)
        # Compute context vector
        context = (alpha.unsqueeze(2) * features).sum(dim=1)  # (B, feature_dim)
        return context, alpha



In [ ]:
class AttentionDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, feature_dim=2048, attn_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attention = BahdanauAttention(hidden_dim, feature_dim, attn_dim)
        
        # Projection layer to match word embedding dimension to the feature dimension
        self.word_proj = nn.Linear(embed_dim, feature_dim)
        
        self.fc = nn.Linear(feature_dim, vocab_size)
        self.hidden_dim = hidden_dim

    def forward(self, features, captions):
        B, T = captions.size()
        device = features.device
        outputs = []
        hidden = torch.zeros(B, hidden_dim, device=device)  # Initialize hidden state
        for t in range(T - 1):
            word_embed = self.embedding(captions[:, t])          # (B, embed_dim)
            word_embed_proj = self.word_proj(word_embed)         # (B, feature_dim)
            context, _ = self.attention(hidden, features)        # (B, feature_dim)
            # Combine the projected word embedding and the attention context
            combined_input = word_embed_proj + context           # (B, feature_dim)
            logit = self.fc(combined_input)                       # (B, vocab_size)
            outputs.append(logit.unsqueeze(1))
        outputs = torch.cat(outputs, dim=1)  # (B, T-1, vocab_size)
        return outputs

    def generate_caption_with_attention(self, features, vocab, max_caption_length=20):
        """
        Generate a caption for a given image, along with the attention weights.

        features: The image features from the encoder (e.g., ResNet)
        vocab: The vocabulary (used for converting word indices to words)
        max_caption_length: The maximum length of the generated caption
        """
        B = features.size(0)  # Batch size
        caption = [vocab.stoi['<sos>']]  # Start with <sos> token
        alphas = []  # To store attention weights

        hidden = torch.zeros(B, self.hidden_dim, device=features.device)

        for _ in range(max_caption_length):  # Generate up to `max_caption_length` words
            word_embed = self.embedding(torch.tensor(caption).to(features.device))  # (B, embed_dim)
            word_embed_proj = self.word_proj(word_embed)  # (B, feature_dim)

            # Compute the attention for the current word
            context, alpha = self.attention(hidden, features)  # (B, feature_dim), (B, 49)

            # Combine the word embedding and context vector
            combined_input = word_embed_proj + context  # (B, feature_dim)

            # Generate the next word
            logits = self.fc(combined_input)  # (B, vocab_size)
            probs = torch.softmax(logits, dim=1)

            word_idx = probs.argmax(dim=1).item()  # Get the word with the highest probability
            caption.append(word_idx)

            # Store the attention weights
            alphas.append(alpha.cpu().detach().numpy())

            # Stop if <eos> token is generated
            if word_idx == vocab.stoi['<eos>']:
                break

        # Convert the word indices to actual words using vocab
        caption_words = [vocab.itos[i] for i in caption]
        return caption_words, alphas


In [ ]:
from nltk.translate.bleu_score import corpus_bleu

def evaluate_bleu(encoder, decoder, dataloader, idx2word, device):
    """
    Runs inference on dataloader, collects references & candidates,
    and returns BLEU-1, BLEU-2, BLEU-3, and BLEU-4 scores.
    """
    encoder.eval()
    decoder.eval()
    refs, hyps = [], []
    
    with torch.no_grad():
        for imgs, caps, _ in tqdm(dataloader, desc="Evaluating", unit="batch"):
            imgs, caps = imgs.to(device), caps.to(device)
            
            # Forward pass
            features = encoder(imgs)
            logits = decoder(features, caps)
            preds = logits.argmax(dim=2)  # Predicted word indices
            
            # Collect reference and hypothesis sequences
            for ref_seq, pred_seq in zip(caps[:, 1:], preds):
                # Ground truth references (skip <sos> and <eos>)
                ref = [idx2word[i.item()] for i in ref_seq if i.item() not in {0, 1}]
                # Hypothesis sequence (skip <sos> and <eos>)
                hyp = [idx2word[i.item()] for i in pred_seq if i.item() not in {0, 1}]
                refs.append([ref])
                hyps.append(hyp)
    
    # Calculate BLEU scores
    bleu_1 = corpus_bleu(refs, hyps, weights=(1, 0, 0, 0))  # BLEU-1
    bleu_2 = corpus_bleu(refs, hyps, weights=(0.5, 0.5, 0, 0))  # BLEU-2
    bleu_3 = corpus_bleu(refs, hyps, weights=(0.33, 0.33, 0.33, 0))  # BLEU-3
    bleu_4 = corpus_bleu(refs, hyps, weights=(0.25, 0.25, 0.25, 0.25))  # BLEU-4
    
    return bleu_1, bleu_2, bleu_3, bleu_4


In [ ]:
class Vocabulary:
    """Simple word-to-index mapping."""
    def __init__(self, freq_threshold=5):
        self.freq_threshold = freq_threshold
        self.itos = {0: "<pad>", 1: "<sos>", 2: "<eos>", 3: "<unk>"}
        self.stoi = {v: k for k, v in self.itos.items()}

    def __len__(self):
        return len(self.itos)

    @staticmethod
    def tokenizer(text):
        return [tok.lower() for tok in nltk.word_tokenize(text)]

    def build_vocab(self, captions_file):
        counter = Counter()
        with open(captions_file, 'r', encoding='utf-8') as f:
            # skip header if present
            first = f.readline()
            if 'caption' in first.lower():
                pass
            else:
                f.seek(0)
            for line in f:
                parts = line.strip().split(',', 1)  # split on first comma
                if len(parts) != 2:
                    continue
                _, caption = parts
                tokens = Vocabulary.tokenizer(caption)
                counter.update(tokens)

        idx = len(self.itos)
        for word, cnt in counter.items():
            if cnt >= self.freq_threshold:
                self.stoi[word] = idx
                self.itos[idx] = word
                idx += 1

    def numericalize(self, text):
        tokens = Vocabulary.tokenizer(text)
        nums = [self.stoi.get(tok, self.stoi["<unk>"]) for tok in tokens]
        return [self.stoi["<sos>"]] + nums + [self.stoi["<eos>"]]



In [ ]:
class Flickr8kDataset(Dataset):
    def __init__(self, images_dir, captions_file, ids, vocab, transform=None):
        """
        ids: either a path to a text file listing image names, or a list of image filenames.
        """
        self.images_dir = images_dir
        self.vocab = vocab
        self.transform = transform

        # Load split IDs
        if isinstance(ids, (list, tuple)):
            self.ids = ids
        else:
            with open(ids, 'r', encoding='utf-8') as f:
                self.ids = [line.strip() for line in f]

        # Load captions (CSV: image,caption)
        self.captions = {}
        with open(captions_file, 'r', encoding='utf-8') as f:
            header = f.readline()
            if 'caption' not in header.lower():
                f.seek(0)
            for line in f:
                parts = line.strip().split(',', 1)
                if len(parts) != 2:
                    continue
                img, caption = parts
                if img in self.ids:
                    self.captions.setdefault(img, []).append(caption.strip())

        # Flatten to (img, caption) pairs
        self.samples = []
        for img, caps in self.captions.items():
            for cap in caps:
                self.samples.append((img, cap))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_name, caption = self.samples[idx]
        img_path = os.path.join(self.images_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        cap_idxs = self.vocab.numericalize(caption)
        cap_tensor = torch.tensor(cap_idxs, dtype=torch.long)
        return image, cap_tensor


# ----------------------------------------------------------------------------- 
# 3. Collate function for padding
# ----------------------------------------------------------------------------- 
def collate_fn(batch):
    images, captions = zip(*batch)
    images = torch.stack(images, 0)
    lengths = [len(c) for c in captions]
    max_len = max(lengths)
    padded = torch.zeros(len(captions), max_len, dtype=torch.long)
    for i, cap in enumerate(captions):
        padded[i, :lengths[i]] = cap
    return images, padded, lengths


# ----------------------------------------------------------------------------- 
# 4. DataLoader factory
# ----------------------------------------------------------------------------- 
def get_loaders(
    root_dir="data/Images",
    captions_file="data/captions.txt",
    train_ids=None,
    val_ids=None,
    test_ids=None,
    freq_threshold=5,
    batch_size=64,
    num_workers=4
):
    # Transforms
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    # Build vocab
    vocab = Vocabulary(freq_threshold)
    vocab.build_vocab(captions_file)

    # Helper to list all image files if no IDs provided
    def load_ids(ids_arg):
        if ids_arg and os.path.isfile(ids_arg):
            with open(ids_arg, 'r', encoding='utf-8') as f:
                return [line.strip() for line in f]
        return [
            fn for fn in os.listdir(root_dir)
            if fn.lower().endswith(('.jpg', 'jpeg', 'png'))
        ]

    train_list = load_ids(train_ids)
    val_list = load_ids(val_ids)
    test_list = load_ids(test_ids)

    # Create datasets
    train_ds = Flickr8kDataset(root_dir, captions_file, train_list, vocab, transform)
    val_ds = Flickr8kDataset(root_dir, captions_file, val_list, vocab, transform)
    test_ds = Flickr8kDataset(root_dir, captions_file, test_list, vocab, transform)

    # DataLoaders
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, collate_fn=collate_fn)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, collate_fn=collate_fn)

    return train_loader, val_loader, test_loader, vocab

In [ ]:
# 1. Set device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Load dataset and DataLoader
root_dir = '/kaggle/input/flickr8k/Images'
captions_file = '/kaggle/input/flickr8k/captions.txt'

# Define the vocab and create DataLoader
train_loader, val_loader, test_loader, vocab = get_loaders(
    root_dir=root_dir, 
    captions_file=captions_file,
    batch_size=64,
    num_workers=4
)


In [ ]:
# Initialize encoder and decoder
vocab_size = len(vocab)
embed_dim = 256
hidden_dim = 512
encoder = ResNetEncoder().to(device)
decoder = AttentionDecoder(vocab_size, embed_dim, hidden_dim).to(device)

In [ ]:
# 4. Initialize optimizer and loss function
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-4)
criterion = nn.CrossEntropyLoss()

In [ ]:
# 5. Training loop
def train_epoch(loader, encoder, decoder, optimizer, criterion, device):
    encoder.train()
    decoder.train()
    running_loss = 0.0
    for images, captions, lengths in tqdm(loader, desc="Training", unit="batch"):
        images, captions = images.to(device), captions.to(device)
        
        # Forward pass
        features = encoder(images)
        outputs = decoder(features, captions)
        
        # Flatten outputs and captions for loss calculation
        outputs = outputs.view(-1, outputs.size(2))  # (B * T-1, vocab_size)
        captions = captions[:, 1:].contiguous().view(-1)  # (B * (T-1))
        
        # Compute loss
        loss = criterion(outputs, captions)
        running_loss += loss.item()
        
        # Backprop and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    avg_loss = running_loss / len(loader)
    return avg_loss

In [ ]:
num_epochs = 1
for epoch in range(num_epochs):
    start_time = time.time()
    train_loss = train_epoch(train_loader, encoder, decoder, optimizer, criterion, device)
    elapsed_time = time.time() - start_time
    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {train_loss:.4f}, Time: {elapsed_time:.2f}s")

In [ ]:
from nltk.translate.bleu_score import corpus_bleu

def evaluate_bleu(encoder, decoder, dataloader, idx2word, device):
    """
    Runs inference on dataloader, collects references & candidates,
    and returns BLEU-1, BLEU-2, BLEU-3, and BLEU-4 scores.
    """
    encoder.eval()
    decoder.eval()
    refs, hyps = [], []
    
    with torch.no_grad():
        for imgs, caps, _ in tqdm(dataloader, desc="Evaluating", unit="batch"):
            imgs, caps = imgs.to(device), caps.to(device)
            
            # Forward pass
            features = encoder(imgs)
            logits = decoder(features, caps)
            preds = logits.argmax(dim=2)  # Predicted word indices
            
            # Collect reference and hypothesis sequences
            for ref_seq, pred_seq in zip(caps[:, 1:], preds):
                # Ground truth references (skip <sos> and <eos>)
                ref = [idx2word[i.item()] for i in ref_seq if i.item() not in {0, 1}]
                # Hypothesis sequence (skip <sos> and <eos>)
                hyp = [idx2word[i.item()] for i in pred_seq if i.item() not in {0, 1}]
                refs.append([ref])
                hyps.append(hyp)
    
    # Calculate BLEU scores
    bleu_1 = corpus_bleu(refs, hyps, weights=(1, 0, 0, 0))  # BLEU-1
    bleu_2 = corpus_bleu(refs, hyps, weights=(0.5, 0.5, 0, 0))  # BLEU-2
    bleu_3 = corpus_bleu(refs, hyps, weights=(0.33, 0.33, 0.33, 0))  # BLEU-3
    bleu_4 = corpus_bleu(refs, hyps, weights=(0.25, 0.25, 0.25, 0.25))  # BLEU-4
    
    return bleu_1, bleu_2, bleu_3, bleu_4


In [ ]:
# Assuming you have the test_loader ready for evaluation
bleu_1, bleu_2, bleu_3, bleu_4 = evaluate_bleu(encoder, decoder, test_loader, vocab.itos, device)

print(f"BLEU-1: {bleu_1:.4f}")
print(f"BLEU-2: {bleu_2:.4f}")
print(f"BLEU-3: {bleu_3:.4f}")
print(f"BLEU-4: {bleu_4:.4f}")
